# Pilot collection

Protocol-matched pilot for the Apertus validity study
(`docs/framework-empirical-evaluation.md` Section 6).

The pilot supplies the eight cell standard deviations of subject-level
relative error. Those SDs are **not** the primary TOSTs. After this
notebook, run `r/power.R` and collect the main sample separately.

Needs CUDA. Do not treat `src/zepto/empirical/` as the study.

**Google Colab:** Runtime → Change runtime type → GPU (T4 or L4). Run the
bootstrap cell first. It clones this branch, installs Zepto without
replacing Colab's CUDA PyTorch, mounts Drive so CSVs survive the VM,
and checks that `torch.cuda` is available.

In [1]:
# Colab / local bootstrap.
# Colab: Runtime → Change runtime type → GPU, then run this cell.
# Local: skip clone and Drive if the repo is already on disk.

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RobinGirardin/zepto.git"
STUDY_BRANCH = "empirical-test"
DRIVE_STUDY_DIR = Path("MyDrive") / "zepto-apertus-validity"


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        marker = candidate / "studies" / "apertus-validity" / "python"
        if marker.is_dir():
            return candidate
    return None


IN_COLAB = in_colab()
REPO_ROOT = find_repo_root(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACTS_ROOT = Path("/content/drive") / DRIVE_STUDY_DIR
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
else:
    ARTIFACTS_ROOT = None

if REPO_ROOT is None:
    if not IN_COLAB:
        raise FileNotFoundError(
            "could not find studies/apertus-validity; open this notebook from the repo"
        )
    clone_dir = Path("/content/zepto")
    if not (clone_dir / "studies" / "apertus-validity" / "python").is_dir():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                STUDY_BRANCH,
                "--depth",
                "1",
                REPO_URL,
                str(clone_dir),
            ]
        )
    REPO_ROOT = clone_dir

STUDY_ROOT = REPO_ROOT / "studies" / "apertus-validity"
if ARTIFACTS_ROOT is None:
    ARTIFACTS_ROOT = STUDY_ROOT / "artifacts"

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(STUDY_ROOT / "python"))

if IN_COLAB:
    # Do not pip-install torch: Colab already has a CUDA build.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"]
    )

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "this notebook needs a CUDA GPU. In Colab: Runtime → Change runtime type → GPU."
    )

print("colab:", IN_COLAB)
print("repo:", REPO_ROOT)
print("study:", STUDY_ROOT)
print("artifacts:", ARTIFACTS_ROOT)
print("gpu:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

Mounted at /content/drive
colab: True
repo: /content/zepto
study: /content/zepto/studies/apertus-validity
artifacts: /content/drive/MyDrive/zepto-apertus-validity
gpu: Tesla T4 (7, 5)


## Draw subjects

A subject is `(configuration_id, seq_len, batch_size)`.
Precision is not sampled. Use a small n for the pilot.

In [2]:
from apertus_validity import Catalog, enumerate_frame, run_collection

catalog = Catalog()
print("frame size:", len(enumerate_frame(catalog)))

N_PILOT = 12
SEED = 1
OUT = ARTIFACTS_ROOT / "pilot"
OUT

frame size: 1944


PosixPath('/content/drive/MyDrive/zepto-apertus-validity/pilot')

## Collect

Process-wide cuBLAS warmup once, then both precisions on each subject.
K = 1 scored training step. Failures go to `ledger.csv`.

In [3]:
run_collection(N_PILOT, seed=SEED, output_dir=OUT, catalog=catalog)
print("wrote", OUT)
print("next: Rscript studies/apertus-validity/r/power.R --evaluation", OUT / "evaluation.csv")

[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=128
[transformers] CUDA-fused xIELU not available (No module named 'xielu') – falling back to a Python version.
For CUDA xIELU (experimental), `pip install git+https://github.com/nickjbrowning/XIELU`
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=128
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=64
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=64
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=64
[transformers] `r

wrote /content/drive/MyDrive/zepto-apertus-validity/pilot
next: Rscript studies/apertus-validity/r/power.R --evaluation /content/drive/MyDrive/zepto-apertus-validity/pilot/evaluation.csv
